# Introduction to AI Agents

If you have used ChatGPT or Claude in a browser, you already know what a **chatbot** feels like: you type a question, the model replies, and the conversation stops until you type again. That pattern is useful for explanation, brainstorming, and drafting text.

An **AI agent** goes further. Instead of only producing language, it runs a **loop**:

1. **Perceive** — read the user goal, prior messages, and any tool results already in context.
2. **Reason** — decide whether to answer directly or call a tool (search docs, create a task, call an API).
3. **Act** — execute that decision in the real world (or a simulated environment).
4. **Observe** — feed the tool result back into context and repeat until the goal is done or a safety limit is hit.

That loop is the defining difference. A chatbot **informs**. An agent **accomplishes** multi-step goals that require decisions and external actions.

The sections immediately below introduce **Agentic AI** as a paradigm: six defining characteristics, five core components, and how plans are evaluated — before we write any code.

---

## What you will build in this notebook

We implement a small but **real** agent — no orchestration framework required — for a fictional **Developer Productivity Assistant**:

- Search an internal documentation knowledge base.
- Create and list tasks on an in-memory task board.
- Run the full perceive → reason → act cycle with guardrails (`max_steps`, request classification).

Everything is self-contained in this notebook. You do not need prior notebooks, shared teaching corpora, or external services to understand the core ideas. An optional cell at the end shows how the same pattern connects to a live LLM when you have an API key.

**Prerequisites:** basic Python (functions, dictionaries, dataclasses). Familiarity with REST APIs is helpful but not required.

---

## What is Agentic AI?

**Agentic AI** is a software paradigm where the system takes a **high-level goal** from a user and works toward achieving it **independently**, with minimal step-by-step human guidance.

Compare that to **reactive Generative AI** — like a standard chatbot. A chatbot waits for your prompt, responds once, and stops. It does not pursue an outcome on its own.

An **agentic system** is different. It is:

- **Proactive** — it monitors progress and takes the next step without you typing again.
- **Self-directed** — it plans its own sequence of actions.
- **Action-oriented** — it executes those actions in the real world (via tools, APIs, databases) until the goal is met or a safety limit is hit.

> **Simple rule:** A chatbot **informs**. An agentic system **accomplishes** a goal through a loop of decisions and actions.

This notebook builds on that idea: first the concepts below, then a working agent you can run in Python.

---

## Six Key Characteristics of Agentic AI

Any system that deserves the label **agentic** can be identified by these six traits. They work together — remove one, and the system starts behaving more like a plain chatbot than an autonomous agent.

### 1. Autonomy

The system can **make decisions and take actions on its own**, without needing a human to approve every step.

- It does not wait for you to say "now do step 2."
- It monitors whether things are working, spots problems, and initiates corrective actions.
- It is **proactive by nature** — the agent drives the work forward, not just the user.

*Example:* If a documentation search returns nothing useful, an autonomous agent retries with a different query instead of giving up after one attempt.

---

### 2. Goal Oriented

The agent operates with a **persistent, high-level objective** — not just a single isolated question.

- Every plan and every action is aimed at reaching that outcome.
- The goal acts like a **compass**: even when the agent calls tools or handles errors, it keeps steering toward the same destination.
- This is what separates "answer this question" from "complete this task."

*Example:* The goal *"Prepare me for production deployment"* might lead the agent to search runbooks, list open tasks, and create a checklist — all in service of one objective.

---

### 3. Planning

The agent can **break a high-level goal into a structured sequence of smaller, actionable steps**.

- It does not jump straight to the final answer; it first figures out *what to do, in what order*.
- A strong planner often generates **multiple candidate plans**, compares them, and picks the best one.
- Plans are typically evaluated on criteria such as **efficiency**, **tool availability**, and **likelihood of success**.

*Example:* For *"Create a deploy task and find rollback steps"*, a plan might be: (1) list existing tasks to avoid duplicates → (2) search docs for rollback guidance → (3) create the task → (4) summarize results.

---

### 4. Reasoning

**Reasoning** is the cognitive work the agent does to interpret information and make decisions — during both planning and execution.

- It decides **which tool** to use and **why**.
- It estimates whether it has **enough information** to finish or needs another step.
- It handles **errors and ambiguity** — e.g., a failed API call or an unclear user request.

Without reasoning, you only have a fixed script. With reasoning, the agent can adapt its next move based on what it just learned.

*Example:* After seeing a duplicate task on the board, the agent reasons that creating another copy is unnecessary and instead reports the existing task.

---

### 5. Adaptability

The agent can **change its strategy or actions** when conditions shift — unexpected errors, new feedback, or updated goals.

- If a tool fails, it tries an alternative path.
- If the environment changes (e.g., low response rate, missing data), it pivots.
- If the **user changes the goal mid-task**, a well-designed agent can incorporate that update rather than blindly finishing the old plan.

> **Can users change agent goals mid-task?**  
> **Yes.** Adaptability explicitly includes responding to changing goals. When a user revises the objective, the agent should detect the update (via context and memory), re-evaluate its plan, and adjust its next actions. Production systems often treat the latest user message as the current goal while retaining history for context.

*Example:* If `search_documentation` returns no results, an adaptable agent broadens the query or asks a clarifying question instead of hallucinating an answer.

---

### 6. Context Awareness

The agent **retains and uses information** from past interactions, user preferences, and the current environment — not just the latest message.

- It knows **where it is in a long-running task** (what already succeeded, what is still pending).
- It remembers **user preferences** (e.g., preferred format, past decisions).
- It tracks **environment state** (open tasks, tool results, session history).

Context awareness is usually implemented through **memory** — short-term (current session) and long-term (history, goals, preferences). Without it, every step would feel like starting from zero.

*Example:* An agent that already listed your open tasks in step 1 does not need to list them again in step 3 unless the board may have changed.

---

## Five Components of an Agentic Application

A real agentic product is not "just an LLM." It is assembled from five building blocks that map cleanly to the characteristics above:

| Component | Role | Maps to |
|-----------|------|---------|
| **Brain** | The LLM that handles reasoning, planning, and language generation | Reasoning, Planning |
| **Orchestrator** | Sequences tasks, routes between steps, manages loops and retries (e.g., LangGraph) | Planning, Autonomy |
| **Tools** | Interfaces to the outside world — APIs, web search, databases, file systems | Autonomy, Goal Oriented |
| **Memory** | Short-term (current session) + long-term (history, goals, user preferences) | Context Awareness |
| **Supervisor** | Human-in-the-loop layer — approvals, guardrails, escalations | Adaptability, Safety |

### Brain

Typically a **Large Language Model (LLM)**. It is the "thinking" layer: it interprets the goal, drafts plans, chooses tools, and writes final responses. It does not touch external systems directly — it decides *what* should happen.

### Orchestrator

Acts like a **project manager** for the agent loop. It decides *when* to call the brain, *when* to run a tool, and *when* to stop. Frameworks such as **LangGraph** often fill this role by modeling the workflow as a state machine or graph.

### Tools

The agent's **hands**. Tools let it search documentation, create records, call APIs, run code, and read files. Without tools, the brain can only talk — it cannot *do* anything in the world.

### Memory

Divided into two layers:

- **Short-term memory** — the current conversation, recent tool results, and working state for this session.
- **Long-term memory** — persisted goals, user preferences, and past interactions across sessions.

Memory is what makes context awareness possible over tasks that span many steps or many days.

### Supervisor

The **safety and governance** layer. Not every action should run fully unattended.

> **What is the supervisor component's role?**  
> The supervisor handles **human-in-the-loop (HITL)** interactions: requesting approval before sensitive actions, enforcing **guardrails** (step limits, allow-listed tools), and **escalating** to a human when the agent is stuck, uncertain, or about to do something risky. Think of it as the manager who can pause the agent and say "wait — let a person confirm this."

---

### How are agentic plans evaluated?

When the brain (or planner) generates candidate plans, they are scored against practical criteria before execution:

1. **Efficiency** — fewer steps and lower cost (tokens, API calls, latency).
2. **Tool availability** — can the required tools actually run with current permissions and data?
3. **Likelihood of success** — does the plan address known constraints from context and memory?
4. **Risk** — does the plan avoid dangerous or irreversible side effects without supervisor approval?

The orchestrator (or the brain itself) selects the **best-scoring plan**, executes it, and re-evaluates after each observation. If reality diverges from the plan — a tool fails, data is missing — adaptability kicks in and a new plan may be generated.

This evaluate → execute → observe → re-evaluate cycle is exactly what you will see in the **perceive → reason → act** loop later in this notebook.

In [ ]:
"""
Shared environment for the Developer Productivity Assistant.
This cell sets up an in-memory knowledge base and task store.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

# Optional: set your key here or in the environment before running live-LLM cells.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

# --- Internal documentation (simulates a company wiki / runbook store) ---
KNOWLEDGE_BASE: list[dict[str, str]] = [
    {
        "id": "deploy-001",
        "title": "Production deployment checklist",
        "text": (
            "Before deploying: run the test suite, verify database migrations with "
            "`alembic upgrade head`, confirm the staging smoke test passed, and "
            "notify #releases in Slack. Roll back using the previous container tag if health checks fail."
        ),
    },
    {
        "id": "auth-002",
        "title": "API authentication",
        "text": (
            "All internal APIs require a JWT bearer token. Send the header "
            "`Authorization: Bearer <token>`. Tokens expire after 8 hours; refresh via /auth/refresh."
        ),
    },
    {
        "id": "onboard-003",
        "title": "New engineer onboarding",
        "text": (
            "Day 1: clone the monorepo, copy `.env.example` to `.env`, run `docker compose up`, "
            "and complete the security training module. Request AWS access through the IT portal."
        ),
    },
    {
        "id": "incident-004",
        "title": "Incident response",
        "text": (
            "For SEV-1 incidents: open a PagerDuty incident, assign an incident commander, "
            "post status updates every 15 minutes in #incidents, and capture a timeline in the postmortem doc."
        ),
    },
    {
        "id": "db-005",
        "title": "Database migrations",
        "text": (
            "Create migrations with Alembic autogenerate after model changes. Never edit applied "
            "revision files. In CI, run migrations against a disposable database before merge."
        ),
    },
]

# --- In-memory task board (simulates a tasks API / database) ---
TASK_BOARD: list[dict[str, Any]] = [
    {
        "id": "task-101",
        "title": "Review open pull requests",
        "status": "open",
        "created_at": "2026-07-08T09:00:00Z",
    },
    {
        "id": "task-102",
        "title": "Update staging environment variables",
        "status": "open",
        "created_at": "2026-07-09T14:30:00Z",
    },
]

print(f"Knowledge articles loaded: {len(KNOWLEDGE_BASE)}")
print(f"Open tasks on board: {sum(1 for t in TASK_BOARD if t['status'] == 'open')}")

---

## 1. Chatbot vs Agent — The Core Distinction

The word *agent* is often overused. A useful rule: if the system only transforms text in one shot, it is probably a chatbot (or a simple LLM pipeline). If it **repeatedly decides and acts** until a goal is met, it is an agent.

| Dimension | **Chatbot** | **AI Agent** |
|-----------|-------------|--------------|
| **Interaction model** | Turn-based Q&A | Goal-directed loop |
| **World access** | Text in → text out | Tools, APIs, files, databases |
| **Memory** | Optional session history | Working memory + tool state + observations |
| **Stopping condition** | User sends the next message | Task complete, budget exhausted, or guardrail |
| **Typical example** | "What is a JWT?" | "Create a deploy task and find the runbook for rollbacks" |

### A concrete contrast

**Chatbot path:** User asks *"How do we authenticate internal APIs?"* → model answers from training data or a single retrieved chunk → done.

**Agent path:** User asks *"Create a task to rotate API tokens and tell me the auth runbook steps."* → agent lists existing tasks (avoid duplicates) → searches docs for JWT guidance → creates the task → summarizes both results → done.

The second request needs **multiple dependent decisions**. That is where agents earn their complexity.

---

## 2. The Perceive–Reason–Act Loop

Every agent architecture — from a 50-line Python script to a multi-agent orchestrator — reduces to the same cycle:

```
  ┌──────────┐     ┌──────────┐     ┌──────────┐
  │ PERCEIVE │ ──► │  REASON  │ ──► │   ACT    │
  │ context  │     │ plan /   │     │ tool /   │
  │ + tools  │     │ decide   │     │ API call │
  └────▲─────┘     └──────────┘     └────┬─────┘
       │                                  │
       └──────── observation / result ◄───┘
```

| Phase | Question the agent answers | Example in our assistant |
|-------|---------------------------|--------------------------|
| **Perceive** | What is the goal? What do I already know? | User wants deploy guidance; two open tasks exist on the board |
| **Reason** | What should I do next? | Search docs for "deployment checklist" before answering |
| **Act** | Execute the decision | Call `search_documentation(query="deployment checklist")` |
| **Observe** | What happened? | Tool returns article `deploy-001` with rollback steps |

The loop may run once (search → answer) or many times (list tasks → create task → verify → summarize). Production agents always cap iterations to control cost and runaway behavior.

In [ ]:
@dataclass
class AgentStep:
    """One line in an agent execution trace."""
    phase: str       # perceive | reason | act | observe
    detail: str
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


def trace_perceive_reason_act(goal: str) -> list[AgentStep]:
    """
    Walk through a realistic agent trace without calling an LLM.
    This makes the loop tangible before we wire up tools.
    """
    return [
        AgentStep(
            "perceive",
            f"User goal: '{goal}'. Task board has {len(TASK_BOARD)} tasks. "
            f"Knowledge base has {len(KNOWLEDGE_BASE)} articles.",
        ),
        AgentStep(
            "reason",
            "Goal mentions deployment — I should search internal docs before answering.",
        ),
        AgentStep(
            "act",
            "Call tool search_documentation(query='production deployment checklist').",
        ),
        AgentStep(
            "observe",
            "Tool returned article deploy-001: run tests, apply migrations, notify #releases.",
        ),
        AgentStep(
            "reason",
            "I have enough context to produce a final answer — no more tools needed.",
        ),
    ]


goal = "What should I do before deploying to production?"
for step in trace_perceive_reason_act(goal):
    print(f"[{step.phase.upper():8}] {step.detail}")

---

## 3. The Agent Stack — From User Interface to the Real World

Agents are not "just the LLM." In production, several layers cooperate:

```
 ┌─────────────────────────────────────────────────────┐
 │  Application / UX (web chat, CLI, IDE plugin)        │
 ├─────────────────────────────────────────────────────┤
 │  Orchestration (state machine, multi-agent routing)    │
 ├─────────────────────────────────────────────────────┤
 │  Agent core (loop, memory, planning, guardrails)     │
 ├─────────────────────────────────────────────────────┤
 │  LLM (language + structured tool-call decisions)     │
 ├─────────────────────────────────────────────────────┤
 │  Tools / APIs (search, CRUD, code execution, MCP)    │
 └─────────────────────────────────────────────────────┘
```

| Layer | Responsibility | In this notebook |
|-------|----------------|------------------|
| **UX** | Collect user goals, show progress | `run_agent(user_message)` prints the trace |
| **Agent core** | Loop, message history, step limits | `DeveloperAssistantAgent` class |
| **LLM** | Choose tools and draft final answers | Rule-based planner (offline) or OpenAI (optional) |
| **Tools** | Grounded actions on data | `search_documentation`, `list_tasks`, `create_task` |

Frameworks (LangGraph, AutoGen, CrewAI, etc.) mainly help with the **orchestration** and **agent core** layers. The loop itself is always the same.

---

## 4. When Agents Help — And When They Do Not

Agents add latency, cost, and failure modes. Use them when the problem genuinely needs iterative decisions.

| Signal | Agent is a good fit | Prefer a simpler approach |
|--------|---------------------|---------------------------|
| **Steps** | 3+ dependent decisions | Single LLM call + template |
| **Data freshness** | Live database or API required | Static context in the prompt |
| **Tool breadth** | Many endpoints or formats | One fixed function |
| **Ambiguity** | User intent unclear; needs clarification | Strict form-based UI |
| **Risk** | Low — internal docs, draft tasks | High — irreversible payments, production deletes |

Our Developer Productivity Assistant is a **safe sandbox**: it reads docs and mutates an in-memory task list. No real production systems are touched, but the control flow mirrors real agents.

---

## 5. Agent Vocabulary

These terms appear throughout agent engineering. Definitions below are self-contained.

| Term | Meaning |
|------|---------|
| **Tool / function** | A named capability the model can invoke (e.g. `create_task`) |
| **Tool schema** | JSON description of the tool's name, purpose, and parameters — sent to the LLM |
| **Tool call** | Structured request from the model: `{name, arguments}` |
| **Observation** | The tool's return value, appended to the conversation as a `tool` message |
| **ReAct** | Pattern that interleaves **Re**asoning (text) and **Act**ing (tool calls) |
| **HITL** | Human-in-the-loop — a person approves sensitive actions before execution |
| **Guardrail** | Hard limit such as `max_steps`, allow-listed tools, or token budget |
| **Trace** | Audit log of messages, tool calls, and timings for debugging |

A **multi-agent system (MAS)** coordinates several specialized agents (planner, researcher, coder). This notebook uses a **single agent with multiple tools** — the simplest useful form.

---

## 6. Implementing the Tools

Tools are ordinary Python functions with clear inputs and outputs. The agent (or LLM) never touches `TASK_BOARD` directly — it calls tools, which keeps permissions and validation in one place.

We implement three tools:

1. **`search_documentation`** — keyword search over the knowledge base.
2. **`list_tasks`** — return open tasks from the board.
3. **`create_task`** — add a new task with duplicate-title protection.

In [ ]:
def search_documentation(query: str, top_k: int = 3) -> list[dict[str, str]]:
    """Return knowledge-base articles ranked by simple keyword overlap."""
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    if not terms:
        return []

    scored: list[tuple[int, dict[str, str]]] = []
    for article in KNOWLEDGE_BASE:
        haystack = f"{article['title']} {article['text']}".lower()
        score = sum(1 for term in terms if term in haystack)
        if score:
            scored.append((score, article))

    scored.sort(key=lambda pair: (-pair[0], pair[1]["id"]))
    return [
        {"id": a["id"], "title": a["title"], "text": a["text"]}
        for _, a in scored[:top_k]
    ]


def list_tasks(status: str = "open") -> list[dict[str, Any]]:
    """List tasks filtered by status (open, done, or all)."""
    if status == "all":
        return list(TASK_BOARD)
    return [t for t in TASK_BOARD if t["status"] == status]


def create_task(title: str) -> dict[str, Any]:
    """Create a new open task; reject duplicate titles (case-insensitive)."""
    title = title.strip()
    if not title:
        raise ValueError("Task title cannot be empty.")

    for existing in TASK_BOARD:
        if existing["title"].lower() == title.lower():
            return {
                "created": False,
                "reason": "duplicate",
                "task": existing,
            }

    task = {
        "id": f"task-{uuid.uuid4().hex[:6]}",
        "title": title,
        "status": "open",
        "created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    TASK_BOARD.append(task)
    return {"created": True, "task": task}


# Quick sanity check — tools work independently of any agent loop
print("Search demo:")
for hit in search_documentation("jwt authentication token"):
    print(f"  [{hit['id']}] {hit['title']}")

print("\nOpen tasks:")
for task in list_tasks("open"):
    print(f"  {task['id']}: {task['title']}")

---

## 7. Tool Schemas — How the LLM Knows What It Can Call

Modern LLM APIs accept a list of **tool schemas** (OpenAI calls them "functions"). Each schema tells the model:

- the tool **name**,
- a natural-language **description** of when to use it,
- the **parameters** it accepts (names, types, which are required).

The model does not execute tools itself. It returns a structured **tool call**; your code runs the function and sends back the result. That separation is what makes tool use safe and testable.

In [ ]:
TOOL_SCHEMAS: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "search_documentation",
            "description": "Search internal engineering documentation for runbooks, policies, and how-to guides.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Keywords describing what to look up.",
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_tasks",
            "description": "List tasks on the team task board, filtered by status.",
            "parameters": {
                "type": "object",
                "properties": {
                    "status": {
                        "type": "string",
                        "enum": ["open", "done", "all"],
                        "description": "Filter tasks by status. Defaults to open.",
                    }
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_task",
            "description": "Create a new open task on the team task board.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "Short imperative title for the task.",
                    }
                },
                "required": ["title"],
            },
        },
    },
]

TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "search_documentation": search_documentation,
    "list_tasks": list_tasks,
    "create_task": create_task,
}

print(f"Registered {len(TOOL_REGISTRY)} tools:")
for name in TOOL_REGISTRY:
    print(f"  - {name}")

---

## 8. The Minimal Agent — Message History + Loop + Tool Executor

Before using any framework, understand the plain-Python shape of an agent:

1. Maintain a **message list** (system, user, assistant, tool).
2. Call the **LLM** with optional tool schemas.
3. If the model returns a **tool call**, execute it and append the **observation**.
4. Repeat until the model returns natural language or `max_steps` is reached.

Below, `RuleBasedPlanner` simulates what an LLM would decide — so the notebook runs fully offline. The agent class itself is the same one you would plug a real LLM into.

In [ ]:
def execute_tool(name: str, arguments: dict[str, Any]) -> str:
    """Dispatch a tool call to the registry and return JSON for the observation."""
    if name not in TOOL_REGISTRY:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        result = TOOL_REGISTRY[name](**arguments)
        return json.dumps(result, default=str)
    except Exception as exc:
        return json.dumps({"error": str(exc)})


class RuleBasedPlanner:
    """
    Offline stand-in for an LLM planner.
    Inspects the user message and returns either a tool_call or final text.
    """

    def plan(self, user_message: str, messages: list[dict[str, Any]], step: int) -> dict[str, Any]:
        text = user_message.lower()

        # If we already have tool results in the trace, synthesize a final answer.
        tool_messages = [m for m in messages if m.get("role") == "tool"]
        if tool_messages and step > 0:
            return {"type": "text", "content": self._summarize(tool_messages, user_message)}

        if any(word in text for word in ("create", "add", "new task")):
            # Extract a rough title after "called" or "named"
            title = self._extract_task_title(user_message)
            return {
                "type": "tool_call",
                "name": "create_task",
                "arguments": {"title": title},
            }

        if any(word in text for word in ("list", "show", "what tasks", "my tasks")):
            return {
                "type": "tool_call",
                "name": "list_tasks",
                "arguments": {"status": "open"},
            }

        # Default: documentation search for how-to / what-is questions
        query = user_message.rstrip("?").strip()
        return {
            "type": "tool_call",
            "name": "search_documentation",
            "arguments": {"query": query},
        }

    def _extract_task_title(self, message: str) -> str:
        for pattern in (
            r"(?:called|named|titled)\s+['\"]?(.+?)['\"]?\s*$",
            r"(?:create|add)\s+(?:a\s+)?task\s+(?:to\s+)?(.+)$",
        ):
            match = re.search(pattern, message, re.IGNORECASE)
            if match:
                return match.group(1).strip(" .'")
        return "Follow up on user request"

    def _summarize(self, tool_messages: list[dict[str, Any]], original_goal: str) -> str:
        last = json.loads(tool_messages[-1]["content"])

        if isinstance(last, list):
            if last and "title" in last[0] and "status" in last[0]:
                lines = [f"- {t['title']} ({t['status']})" for t in last]
                return "Open tasks on your board:\n" + "\n".join(lines)
            if last and "text" in last[0]:
                top = last[0]
                return f"From [{top['id']}] {top['title']}: {top['text']}"

        if isinstance(last, dict) and "task" in last:
            task = last["task"]
            if last.get("created"):
                return f"Created task {task['id']}: '{task['title']}'."
            return f"Task already exists: {task['id']} — '{task['title']}'."

        return f"Completed actions for: {original_goal}"


@dataclass
class DeveloperAssistantAgent:
    """Framework-free single-agent loop with guardrails."""

    planner: RuleBasedPlanner = field(default_factory=RuleBasedPlanner)
    messages: list[dict[str, Any]] = field(default_factory=list)
    trace: list[AgentStep] = field(default_factory=list)
    max_steps: int = 5

    def run(self, user_message: str) -> str:
        self.messages = [
            {
                "role": "system",
                "content": (
                    "You are a developer productivity assistant. Use tools to search "
                    "documentation and manage tasks. Be concise and cite article IDs."
                ),
            },
            {"role": "user", "content": user_message},
        ]
        self.trace = []

        for step in range(self.max_steps):
            self.trace.append(AgentStep("perceive", f"Step {step + 1}: context has {len(self.messages)} messages."))

            decision = self.planner.plan(user_message, self.messages, step)

            if decision["type"] == "tool_call":
                name = decision["name"]
                args = decision["arguments"]
                self.trace.append(AgentStep("reason", f"Decided to call {name}({args})."))
                self.trace.append(AgentStep("act", f"Executing {name}..."))

                observation = execute_tool(name, args)
                self.messages.append({"role": "assistant", "tool_calls": [{"name": name, "arguments": args}]})
                self.messages.append({"role": "tool", "content": observation})
                self.trace.append(AgentStep("observe", observation[:120] + ("..." if len(observation) > 120 else "")))
                continue

            # Final natural-language response
            self.trace.append(AgentStep("reason", "Enough information — returning final answer."))
            answer = decision["content"]
            self.messages.append({"role": "assistant", "content": answer})
            return answer

        return "Stopped: max_steps reached without a final answer."

---

## 9. Running the Agent — Three Real Requests

Each example below exercises a different tool path. Watch the printed trace to see perceive → reason → act → observe in action.

In [ ]:
def run_agent_demo(user_message: str) -> None:
    print("=" * 72)
    print(f"USER: {user_message}")
    print("-" * 72)

    agent = DeveloperAssistantAgent(max_steps=3)
    answer = agent.run(user_message)

    for step in agent.trace:
        print(f"[{step.phase.upper():8}] {step.detail}")

    print("-" * 72)
    print(f"ASSISTANT: {answer}")
    print()


run_agent_demo("How do we authenticate internal APIs?")
run_agent_demo("List my open tasks")
run_agent_demo("Create a task called Rotate JWT signing keys")

---

## 10. Should This Request Use an Agent?

Not every user message needs a full agent loop. A lightweight **router** can send simple questions to a cheap single-turn path and reserve the loop for action-oriented requests.

Below is a deliberately simple classifier. Production systems often use a small model or embedding similarity instead of keyword rules — but the idea is the same: **match capability to complexity**.

In [ ]:
REQUESTS = [
    "What is a JWT?",
    "Create a task called Sprint retro notes",
    "Delete all production databases",
    "Summarize every open task and the deploy runbook",
    "What is 2 + 2?",
]


def classify_request(text: str) -> str:
    """Route to chatbot (single turn) or agent (tool loop)."""
    t = text.lower()

    if any(w in t for w in ("delete", "drop", "production")) and any(
        w in t for w in ("all", "database", "db")
    ):
        return "block"  # dangerous — do not automate

    if any(w in t for w in ("create", "add", "list", "summarize", "run", "deploy")):
        return "agent"

    return "chatbot"


for request in REQUESTS:
    route = classify_request(request)
    print(f"{route:8} → {request}")

---

## 11. Safety Guardrails — Non-Negotiable in Production

| Risk | What goes wrong | Mitigation |
|------|-----------------|------------|
| **Unbounded loops** | Agent calls tools forever; cost explodes | `max_steps`, token budget, wall-clock timeout |
| **Dangerous tools** | Model deletes data or sends money | Allow-listed tools, scoped credentials, HITL approval |
| **Prompt injection via tools** | Untrusted web page text hijacks the agent | Separate trust zones; validate tool outputs |
| **Duplicate side effects** | Same task created twice | Idempotency keys; duplicate checks (see `create_task`) |
| **Silent failures** | Tool errors swallowed; model hallucinates success | Return structured errors; log every observation |

Our agent already implements two guardrails: **`max_steps`** on the loop and **duplicate detection** in `create_task`. The classifier above adds a third: **blocking** overtly dangerous requests.

In [ ]:
def run_bounded_loop(max_steps: int = 3) -> list[str]:
    """Illustrate why max_steps matters — agent cannot spin forever."""
    trace = []
    for i in range(max_steps):
        trace.append(f"step {i + 1}: tool call simulated")
    trace.append("guardrail: max_steps reached — stopping")
    return trace


for line in run_bounded_loop(3):
    print(line)

---

## 12. Observability — Traces Are Your Audit Log

When an agent misbehaves in production, you need to answer: *Which tools fired? In what order? With what arguments? How long did each step take?*

The `agent.messages` list is the seed of a **trace**. Mature systems export traces to OpenTelemetry, Langfuse, or similar platforms. Even in a tutorial, printing role counts helps verify the loop shape.

In [ ]:
def trace_summary(messages: list[dict[str, Any]]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for message in messages:
        role = message.get("role", "unknown")
        counts[role] = counts.get(role, 0) + 1
    return counts


# Re-run one request and inspect the message structure
agent = DeveloperAssistantAgent()
agent.run("What is the incident response process?")

print("Message role counts:", trace_summary(agent.messages))
print("\nFull message trace:")
for i, msg in enumerate(agent.messages):
    preview = json.dumps(msg, default=str)
    print(f"  [{i}] {preview[:100]}{'...' if len(preview) > 100 else ''}")

---

## 13. Single-Agent vs Multi-Agent — A Preview

Most products start with **one agent and many tools** — exactly what we built. You split into multiple agents only when:

- prompts become too large to maintain,
- specialists need different models or temperatures,
- teams want parallel workstreams (researcher + writer + reviewer).

```
Single agent + many tools          Multi-agent system
        │                                  │
   one planner loop                  planner + specialists
        │                                  │
   tool router in one prompt          message passing / supervisor
```

The perceive–reason–act loop does not change. Multi-agent systems add **coordination** on top.

---

## 14. Optional — Connecting a Live LLM

The agent loop is identical when a real model makes planning decisions. Set `USE_LIVE_LLM = True` and provide a valid `OPENAI_API_KEY` to see a one-shot LLM explanation of the perceive–reason–act pattern.

In a full implementation, you would replace `RuleBasedPlanner` with a function that calls `client.chat.completions.create(..., tools=TOOL_SCHEMAS)` and parses `tool_calls` from the response. The `DeveloperAssistantAgent` class would stay the same.

In [ ]:
USE_LIVE_LLM = False  # Set True after setting a real OPENAI_API_KEY

OFFLINE_EXPLANATION = (
    "Perceive–reason–act means: gather context (perceive), decide the next action (reason), "
    "execute a tool or return an answer (act), then feed results back (observe) until done."
)

if USE_LIVE_LLM:
    try:
        from openai import OpenAI

        client = OpenAI()
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Explain AI agents clearly in 2-3 sentences."},
                {"role": "user", "content": "What is the perceive-reason-act loop?"},
            ],
            max_tokens=120,
        )
        print(response.choices[0].message.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print(OFFLINE_EXPLANATION)

---

## 15. Common Beginner Mistakes

| Mistake | Why it hurts | Better approach |
|---------|--------------|------------------|
| Calling every LLM feature an "agent" | Over-engineering simple pipelines | Reserve the term for perceive–reason–act loops with tools |
| Letting the model execute tools directly | No validation, no auth, no logging | Your code dispatches tools via a registry |
| Skipping tool schemas | Fragile regex on free-form JSON | Structured schemas the API understands |
| No `max_steps` | Runaway cost and infinite loops | Hard cap + monitoring alerts |
| Using an agent for static FAQs | Extra latency and cost | Cache answers or use retrieval-only RAG |
| Ignoring observability | Cannot debug production failures | Persist `messages` / traces from day one |

---

## 16. Check Your Understanding

**Concepts (Agentic AI framework):**

1. What is the difference between **reactive Generative AI** (a chatbot) and **Agentic AI**?
2. Name all **six characteristics** of an agentic system and explain one in your own words.
3. What are the **five components** of an agentic application, and which component handles human-in-the-loop approvals?
4. How are **agentic plans evaluated** before execution? What happens when a plan fails mid-task?
5. Can a user **change the agent's goal mid-task**? Which characteristic makes that possible?

**Implementation (this notebook):**

6. Name the four phases of a full agent cycle (include **observe** after **act**).
7. Why does the agent call `list_tasks` before `create_task` in some real systems, even though our simple planner does not always do that?
8. What is the difference between a **tool schema** and a **tool call**?
9. Give one user request that should route to `chatbot` and one that should route to `agent` in our classifier.
10. What two guardrails did we implement in `DeveloperAssistantAgent` and `create_task`?

---

## 17. Summary

**Agentic AI** takes a high-level goal and works toward it independently — proactive, self-planned, and action-oriented. Unlike a **chatbot** that answers one turn at a time, an agent **accomplishes** multi-step goals through a **perceive → reason → act → observe** loop.

**Conceptual foundation:**

- **Six characteristics:** Autonomy, Goal Oriented, Planning, Reasoning, Adaptability, Context Awareness — together they define what makes a system truly agentic.
- **Five components:** Brain (LLM), Orchestrator, Tools, Memory, Supervisor — the building blocks of any real agentic application.
- **Plan evaluation:** Candidate plans are scored on efficiency, tool availability, success likelihood, and risk; the loop re-evaluates after every observation.

**From this notebook's implementation:**

- The **agent stack** spans UX → orchestration → agent core → LLM → tools. Frameworks help with orchestration; the loop is always the same.
- **Tools** are regular functions behind a registry. The LLM returns structured calls; your code executes them and appends observations.
- A **minimal agent** is a message list + planner + tool executor + `max_steps` guardrail. We built `DeveloperAssistantAgent` end to end with working documentation search and task management.
- **Route** simple questions to lightweight paths; reserve full agent loops for action-oriented requests.
- **Traces** (`messages`, `AgentStep` logs) are essential for debugging and trust.

You now have a concrete mental model and runnable code. The next step in your learning is to swap `RuleBasedPlanner` for a real LLM tool-calling API while keeping the same loop structure.